# Chapter 8: Scanning and Enumeration

> "You can't protect what you can't see, and you can't attack what you haven't found." anonymous

---

## Learning Objectives

After completing this chapter, you will be able to:

1. Explain how TCP and UDP port scanning works at the packet level.
2. Describe Nmap scan types and select the appropriate type for a given scenario.
3. Interpret Nmap output including port states, service versions, and OS fingerprints.
4. Use Nmap NSE scripts for targeted enumeration.
5. Enumerate SMB shares, user accounts, and Windows information.
6. Enumerate web application structure using directory brute-forcing.
7. Explain how to reduce scan detectability and the detection mechanisms that counter this.
8. Document scan results as part of a formal penetration test.

## Key Terms

- **Port scan**: probing a host to determine which ports are open, closed, or filtered.
- **SYN scan (half-open)**: sends SYN, reads response, sends RST without completing handshake.
- **Service version detection**: querying an open port to identify the running application and version.
- **OS fingerprinting**: deducing the operating system from network stack behaviour.
- **NSE**: Nmap Scripting Engine; Lua scripts that extend Nmap's enumeration capability.
- **Banner grabbing**: capturing the service identification string sent by an open port.
- **Filtered port**: a port where all probes are dropped by a firewall (no response).
- **SMB**: Server Message Block; Windows file and printer sharing protocol.
- **SNMP**: Simple Network Management Protocol; device management, often misconfigured.
- **Directory brute-forcing**: testing URL paths against a wordlist to find hidden content.

---

## Port Scanning Fundamentals

### Why Port Scanning Is Fundamental

Every network service listens on a port. Knowing which ports are open reveals which services are
running, which reveals potential attack surface. A web server on port 443 is expected; an RDP
service (3389) exposed to the internet or a Telnet service (23) anywhere are findings.

### TCP Port States

Nmap classifies ports into six states:

| State | Meaning |
|---|---|
| Open | A service is actively accepting connections |
| Closed | No service listening; host responded with RST |
| Filtered | No response; firewall is dropping packets |
| Unfiltered | Port accessible but state cannot be determined |
| Open/Filtered | Nmap cannot distinguish open from filtered (UDP) |
| Closed/Filtered | Nmap cannot distinguish closed from filtered |

### The SYN Scan

A TCP SYN scan (nmap -sS) sends a SYN packet and evaluates the response:
- SYN-ACK received: port is open (service is listening); Nmap sends RST to avoid completing the handshake.
- RST received: port is closed.
- No response or ICMP port-unreachable: port is filtered.

The SYN scan does not complete the TCP handshake, so it often does not appear in application logs
(which log at the application layer after accept()). It does appear in network-level logs and IDS.

#### Connect Scan

`nmap -sT` completes the full three-way handshake. It is noisier but can be run without root
privileges and is the fallback when raw socket access is unavailable (e.g., from an unprivileged
user on Windows).

### UDP Scanning

UDP services (DNS port 53, DHCP port 67/68, SNMP port 161, TFTP port 69) do not respond to SYN
packets. Nmap's UDP scan (`-sU`) sends an empty UDP datagram (or a protocol-specific probe) and
interprets:
- UDP response: port is open.
- ICMP port-unreachable: port is closed.
- No response: port is open or filtered.

UDP scanning is slow and unreliable; rate-limit rules on most operating systems cause many responses
to be delayed or dropped, producing false open/filtered results.

---

## Nmap in Depth

### Common Scan Options

| Option | Purpose |
|---|---|
| `-sS` | SYN scan (default with root) |
| `-sV` | Service version detection |
| `-O` | OS fingerprinting |
| `-A` | Aggressive: -sV -O -sC --traceroute |
| `-p 1-65535` | Scan all ports (default: top 1000) |
| `-T0` to `-T5` | Timing: 0=paranoid/slow, 5=insane/fast |
| `-oN/-oX/-oG` | Output: normal/XML/greppable |
| `--open` | Only show open ports |
| `-Pn` | Skip host discovery; treat host as up |

### Service Version Detection

`-sV` sends application-specific probes to open ports and matches responses against Nmap's service
fingerprint database (nmap-service-probes). The output includes the service name, version number,
and any extra information (product, extrainfo). Example output:
```
443/tcp open  ssl/http  Apache httpd 2.4.52 (Ubuntu)
22/tcp  open  ssh       OpenSSH 8.2p1 Ubuntu 4ubuntu0.4
3306/tcp open  mysql    MySQL 8.0.28-0ubuntu0.20.04.3
```
Each version string can be cross-referenced with the NVD or Exploit-DB for known CVEs.

### OS Fingerprinting

Nmap's OS detection (`-O`) sends a series of probes and compares responses against a database of
TCP/IP stack implementations. It identifies the OS and version (Windows Server 2019, Linux 5.15,
FreeBSD 13.0) with a confidence percentage. Accuracy decreases behind NAT, load balancers, or
normalising firewalls that modify packet fields.

### Nmap Scripting Engine

NSE extends Nmap with ~600 scripts organised into categories: auth, broadcast, brute, default,
discovery, dos (avoid in tests), exploit, external, fuzzer, intrusive, malware, safe, version, vuln.

Important scripts for enumeration:

| Script | Purpose |
|---|---|
| `http-title` | Retrieve page title of web services |
| `http-robots.txt` | Retrieve robots.txt |
| `ssl-cert` | Extract TLS certificate details |
| `smb-enum-shares` | List SMB shares |
| `smb-enum-users` | Enumerate Windows users via SMB |
| `dns-zone-transfer` | Attempt zone transfer |
| `snmp-brute` | Brute-force SNMP community strings |
| `mysql-empty-password` | Test for blank MySQL root password |

---

## Service-Specific Enumeration

### SMB Enumeration

Windows networks expose a rich enumeration surface via SMB (port 445). Tools:
- `smbclient -L //target`: list shares.
- `enum4linux`: comprehensive Windows/Samba enumeration including users, groups, password policy.
- `crackmapexec smb target`: fast bulk SMB enumeration across a subnet.

Particularly valuable findings: `ADMIN$` and `C$` shares accessible without authentication (null
session), password policy that allows short or simple passwords, and a list of valid usernames for
password spray attacks.

### SNMP Enumeration

SNMP uses community strings as authentication. The default `public` (read) and `private`
(read-write) strings are set on countless devices. `onesixtyone` rapidly tests community strings
across a subnet. `snmpwalk -v2c -c public target` walks the entire MIB, revealing: system
description (OS and version), interfaces, routing table, running processes, installed software,
user accounts on some implementations, and ARP cache (revealing other hosts on the network).

### Web Directory Brute-Forcing

Tools like `gobuster dir`, `ffuf`, and `dirb` enumerate URL paths by trying each word from a
wordlist. A well-chosen wordlist (SecLists `Discovery/Web-Content/`) finds: admin panels, backup
files (`backup.zip`, `db.sql`), configuration files (`.env`, `config.php`), version control
artefacts (`.git/`, `.svn/`), and API endpoints not linked from the front end.

---

## Detection and Evasion

### How IDS Detects Scanning

Signature-based IDS (Snort, Suricata) matches packet patterns: SYN to multiple ports in sequence is
a classic scan signature. Anomaly-based detection flags high connection rates. Host-based firewalls
log connection attempts. A realistic tester must assume detection is possible and document the scan
rather than trying to hide it in a legitimate engagement.

### Timing and Fragmentation

`-T0` and `-T1` insert long delays between probes, reducing the rate of detected scan events.
Decoy scan (`-D RND:10`) sends probes from multiple spoofed source addresses alongside the real
source, making attribution harder. Packet fragmentation (`-f`) splits packets below the IDS
reassembly threshold. These techniques are relevant for red team exercises; standard penetration
tests operate within defined time windows and do not require evasion.

---

## Why This Matters

A misconfigured service that allows enumeration (SNMP with default community, SMB null session,
an exposed admin panel) is often the first foothold in a network compromise. Scanning reveals
the technology landscape, version numbers, and misconfigurations; enumeration extracts user names,
shares, and policy details that feed every subsequent attack phase. Defenders who run their own
scanning on their networks on a regular schedule find these exposures before attackers do.

---

## News in Focus

Automated internet-wide scanners including those operated by Shodan, Censys, and security researchers
routinely find SNMP devices with default community strings exposing full device configuration, RDP
services with no MFA exposed to the internet, and database servers (MongoDB, Elasticsearch,
Redis) without authentication accessible from the public internet. These are enumeration findings;
the attacker does not need to exploit a vulnerability, because the access is unauthenticated by
design error.

---


In [1]:
# Chapter 8 -- Worked Example: Nmap output parser and port risk scorer

def parse_nmap_output(raw):
    # Parse simplified nmap output lines into structured records.
    findings = []
    for line in raw.strip().splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        parts = line.split()
        if len(parts) >= 3 and '/' in parts[0]:
            port_proto = parts[0]
            state = parts[1]
            service = ' '.join(parts[2:])
            port = int(port_proto.split('/')[0])
            findings.append({'port': port, 'state': state, 'service': service})
    return findings

HIGH_RISK_PORTS = {
    21: "FTP: cleartext, often weak creds",
    23: "Telnet: cleartext protocol, deprecated",
    25: "SMTP open relay risk if misconfigured",
    110: "POP3: cleartext email retrieval",
    135: "MSRPC: Windows attack surface",
    139: "NetBIOS: Windows enumeration",
    445: "SMB: ransomware spread vector",
    1433: "MSSQL: database direct exposure",
    3306: "MySQL: database direct exposure",
    3389: "RDP: brute-force and BlueKeep target",
    5900: "VNC: often no/weak auth",
    6379: "Redis: often unauthenticated by default",
    27017: "MongoDB: often unauthenticated by default",
}

simulated_output = (
    "80/tcp   open  http       Apache httpd 2.4.52 (Ubuntu)\n"
    "443/tcp  open  ssl/https  Apache httpd 2.4.52\n"
    "22/tcp   open  ssh        OpenSSH 8.2p1 Ubuntu\n"
    "3389/tcp open  ms-wbt-server Microsoft Terminal Services\n"
    "445/tcp  open  microsoft-ds Windows Server 2019\n"
    "3306/tcp open  mysql      MySQL 8.0.28\n"
    "6379/tcp open  redis      Redis key-value store\n"
    "21/tcp   closed ftp\n"
)

findings = parse_nmap_output(simulated_output)
print(f"{'Port':<8} {'State':<9} {'Service':<35} {'Risk Note'}")
print("-" * 90)
for f in sorted(findings, key=lambda x: x['port']):
    risk = HIGH_RISK_PORTS.get(f['port'], "")
    print(f"{f['port']:<8} {f['state']:<9} {f['service']:<35} {risk}")

high_risk = [f for f in findings if f['port'] in HIGH_RISK_PORTS and f['state']=='open']
print(f"\n  {len(high_risk)} high-risk open port(s) detected -- investigate immediately.")


Port     State     Service                             Risk Note
------------------------------------------------------------------------------------------
21       closed    ftp                                 FTP: cleartext, often weak creds
22       open      ssh OpenSSH 8.2p1 Ubuntu            
80       open      http Apache httpd 2.4.52 (Ubuntu)   
443      open      ssl/https Apache httpd 2.4.52       
445      open      microsoft-ds Windows Server 2019    SMB: ransomware spread vector
3306     open      mysql MySQL 8.0.28                  MySQL: database direct exposure
3389     open      ms-wbt-server Microsoft Terminal Services RDP: brute-force and BlueKeep target
6379     open      redis Redis key-value store         Redis: often unauthenticated by default

  4 high-risk open port(s) detected -- investigate immediately.


## Review Questions (MCQ)

**Q1.** An Nmap SYN scan sends a SYN and receives SYN-ACK. What does Nmap send next?
A. ACK to complete the connection  B. RST to abort without completing the handshake  C. FIN to close  D. Nothing

**Q2.** A port returning ICMP port-unreachable in response to a UDP probe is:
A. Open  B. Open/Filtered  C. Closed  D. Filtered

**Q3.** Service version detection (`-sV`) works by:
A. Guessing based on port number  B. Sending application probes and matching against a fingerprint database  C. Reading the /etc/services file  D. Running a vulnerability scan

**Q4.** SNMP enumeration is most valuable when:
A. The community string is "private"  B. Default community strings like "public" are in use  C. SNMPv3 is configured  D. The device is running Windows

**Q5.** Gobuster dir is used to:
A. Scan network ports  B. Brute-force URL paths against a wordlist  C. Enumerate SMB shares  D. Perform password spraying

**Q6.** Which Nmap flag scans all 65535 ports?
A. `-sA`  B. `-p-`  C. `-A`  D. `--all-ports`

**Q7.** The Nmap `-T0` timing template is named:
A. Insane  B. Aggressive  C. Paranoid  D. Sneaky

**Q8.** SMB null session enumeration works by:
A. Exploiting a buffer overflow  B. Connecting without credentials to enumerate users and shares  C. Cracking the LM hash  D. Sending a malformed RPC request

**Q9.** An open port 6379 with no authentication is most likely running:
A. MySQL  B. MongoDB  C. Redis  D. Elasticsearch

**Q10.** Which Nmap output format is easiest to process with automated tools?
A. `-oN` (normal)  B. `-oX` (XML)  C. `-oG` (greppable)  D. Screen output

*Answers: Q1 B, Q2 C, Q3 B, Q4 B, Q5 B, Q6 B, Q7 C, Q8 B, Q9 C, Q10 B.*

## Lab Assignment

**Part A -- Self-scan**: Run `nmap -sV -O -p- localhost` on your own machine. Document: all open ports, service names and versions, and OS detection accuracy. For any service running an old version, search the NVD for known CVEs.

**Part B -- Nmap NSE**: Run `nmap --script smb-enum-shares,smb-enum-users <target_IP>` against a Windows VM or lab target you are authorised to test. Document the shares found and whether any are accessible without credentials.

**Part C -- SNMP walk**: If a lab switch or router supports SNMP with a known community string, run `snmpwalk -v2c -c public <target>`. Document: system description, interfaces, and any user or process information revealed.

**Part D -- Web enumeration**: Run `gobuster dir -u http://localhost -w /usr/share/seclists/Discovery/Web-Content/common.txt` against a deliberately vulnerable web application (DVWA, Juice Shop). Document: all discovered paths, the HTTP status code for each, and which three are most interesting from a security perspective.

## References

```{bibliography}
:filter: docname in docnames
```


```{index} Port scan, SYN scan, Service version detection, OS fingerprinting, NSE, Banner grabbing, Filtered port, SMB, SNMP, Directory brute-forcing
```
